In [1]:
import pandas as pd

file_loc = '/mnt/disks/filedisk2a/Qingcheng/'
train_df   = pd.read_csv(file_loc + 'train_df.csv')
holdout_df = pd.read_csv(file_loc + 'holdout_df.csv')
# exposure = pd.read_csv(file_loc + 'exposure_dthr.csv')
# # # stream bid_dthr.csv in chunks, keep only short_profit > 0 (avoids loading the whole file)
# bids = pd.concat(
#     (chunk[chunk['short_profit'] < 0]
#      for chunk in pd.read_csv(file_loc + 'm_dart.csv', chunksize=500_000)),
#     ignore_index=True,
# )
print('train_df  :', train_df.shape)
print('holdout_df:', holdout_df.shape)
train_df.head()

train_df  : (603721, 80)
holdout_df: (231724, 80)


,constraint_family_num,dt,profit,profit_congestion,long_profit,short_profit,long_profit_sum_3d,short_profit_sum_3d,long_profit_sum_7d,short_profit_sum_7d,...,rt_nz_cnt_1y,rt_nz_cnt_3y,rt_spike_7d,rt_spike_1m,rt_spike_3m,rt_spike_1y,rt_spike_3y,KV,KV_group,monitored_name
0,7,2021-07-01,14353.11877,5089.66448,0.0,0.0,0.00000,0.0,0.000000,0.000000,...,0.0,3.0,0.0,0.000000,0.000000,0.000000,-852.205775,161.0,04_345,ln70_bluf-sub1214
1,11,2021-07-01,3893.43353,-828.00122,0.0,0.0,0.00000,0.0,0.000000,0.000000,...,5.0,5.0,0.0,-2829.423183,-2894.467394,-1825.074444,-1841.973281,161.0,04_345,lnadabell-vbi5
2,20,2021-07-01,2606.85200,82.42028,0.0,0.0,-91.79301,0.0,-1198.835983,224.251705,...,1.0,1.0,0.0,0.000000,-876.909880,-884.333313,-885.952971,NaN,03_138,lnallen2-quaker
3,27,2021-07-01,2922.76579,160.79166,0.0,0.0,0.00000,0.0,0.000000,0.000000,...,0.0,30.0,0.0,0.000000,0.000000,0.000000,-909.199441,NaN,03_138,lnanadarko-pocaset
4,28,2021-07-01,10621.29462,2009.66464,0.0,0.0,0.00000,0.0,0.000000,0.000000,...,0.0,1.0,0.0,0.000000,0.000000,0.000000,-228.905063,138.0,03_138,lnanadarko-sequoyah


In [2]:
# top-3 worst short_profit rows per worst-congestion day, across train_df + holdout_df
# 1) the 20 worst congestion-loss days
# 10 worst short_profit (dt, family) from train_df
top10 = (train_df.sort_values('rtm')[['dt','profit_congestion','short_profit','monitored_name','rtm','dam','FlowRatio','KV','ShadowPrice_sum']]
                 .head(120))
# # align key name + dt dtype, then query m_dart
# keys = top10[['dt', 'constraint_family_num']].rename(
#     columns={'constraint_family_num': 'constraintFamilyNum'})
# keys['dt']    = keys['dt'].astype(str)
# m_dart['dt']  = m_dart['dt'].astype(str)

# detail = m_dart.merge(keys, on=['dt', 'constraintFamilyNum'], how='inner')
# detail2 = detail[detail['short_profit']<-100000]

# display(top10)                                              # the 10 (dt, family) + their short_profit
# display(detail2.sort_values(['dt', 'constraintFamilyNum']))  # node-level rows behind them
top10

,dt,profit_congestion,short_profit,monitored_name,rtm,dam,FlowRatio,KV,ShadowPrice_sum
172135,2023-03-01,158887.49025,-1.380617e+06,xfmrforman-forman,-38301.5906,-4795.2005,0.739720,230.0,0.000000
38515,2021-11-12,-1730.28192,-1.197112e+05,lnnash_mps-lbrtywt5,-33641.8508,0.0000,0.858780,161.0,0.000000
289481,2024-01-21,21455.66999,0.000000e+00,lnaur1241-rdspg5,-31876.8579,0.0000,0.909322,161.0,0.000000
147999,2022-12-02,65346.20741,-5.045229e+05,lncalf-apct,-31002.9590,0.0000,0.628008,161.0,0.000000
455639,2025-04-14,13124.13429,-5.768845e+05,lnosage_og-webbtap4,-29471.4135,-16540.7202,1.013335,138.0,-9397.946018
...,...,...,...,...,...,...,...,...,...
537757,2025-11-18,5067.21767,-3.815579e+05,lndenver_c-higgeast,-16636.5763,0.0000,0.647747,115.0,0.000000
436098,2025-03-07,-24220.31986,-5.589204e+05,xfmredwv-edwv,-16569.2133,-10583.8823,1.057478,161.0,-19898.471068
69865,2022-03-29,4350.92111,-5.915650e+04,lnmstng1-sw51,-16521.0399,-8658.5619,0.962946,138.0,0.000000
381772,2024-09-18,9843.86558,-5.694873e+05,xfmrcimarron-cimarron,-16484.0605,-13993.9338,1.185958,345.0,-37426.065514


In [3]:
import numpy as np, pandas as pd

QS  = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
neg = np.isfinite(train_df['short_profit']) & (train_df['short_profit'] < 0)

probs = [0.0, *QS, 1.0]
edges = np.unique(train_df.loc[neg, 'short_profit'].quantile(probs).values)   # dedup tied edges
labels = [f'p{lo*100:g}-p{hi*100:g}' for lo, hi in zip(probs[:-1], probs[1:])][:len(edges)-1]

train_df['short_profit_q'] = pd.cut(
    train_df['short_profit'].where(neg), bins=edges, labels=labels, include_lowest=True)

print(train_df['short_profit_q'].value_counts(dropna=False).sort_index())


QS  = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
neg = np.isfinite(train_df['rt_da']) & (train_df['rt_da'] < 0)

probs  = [0.0, *QS, 1.0]
edges  = np.unique(train_df.loc[neg, 'rt_da'].quantile(probs).values)   # dedup tied edges
labels = [f'p{lo*100:g}-p{hi*100:g}' for lo, hi in zip(probs[:-1], probs[1:])][:len(edges)-1]

train_df['rt_da_q'] = pd.cut(
    train_df['rt_da'].where(neg), bins=edges, labels=labels, include_lowest=True)

print(train_df['rt_da_q'].value_counts(dropna=False).sort_index())
# keep only rows with a band on BOTH sides, then make each short_profit column sum to 100%
sub = train_df.dropna(subset=['rt_da_q', 'short_profit_q'])
ct  = pd.crosstab(sub['rt_da_q'], sub['short_profit_q'])
print((ct.div(ct.sum(0), axis=1) * 100).round(1))   # column %: each short_profit band sums to 100

QS  = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
neg = train_df['short_profit'] < 0
print(train_df.loc[neg, 'short_profit'].quantile([0, *QS, 1]).round(1))

QS  = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
neg = train_df['rt_da'] < 0
print(train_df.loc[neg, 'rt_da'].quantile([0, *QS, 1]).round(3))




short_profit_q
p0-p1          219
p1-p5          876
p5-p10        1095
p10-p30       4380
p30-p50       4380
p50-p70       4379
p70-p90       4380
p90-p95       1095
p95-p99        876
p99-p100       219
NaN         581822
Name: count, dtype: int64
rt_da_q
p0-p1          229
p1-p5          914
p5-p10        1143
p10-p30       4571
p30-p50       4571
p50-p70       4571
p70-p90       4571
p90-p95       1143
p95-p99        914
p99-p100       229
NaN         580865
Name: count, dtype: int64
short_profit_q  p0-p1  p1-p5  p5-p10  p10-p30  p30-p50  p50-p70  p70-p90  \
rt_da_q                                                                    
p0-p1            23.7    7.3     3.3      0.6      0.5      0.2      0.1   
p1-p5            42.9   28.0    13.6      5.6      1.8      0.9      0.4   
p5-p10           21.5   21.3    19.0      9.5      3.1      1.4      0.8   
p10-p30          11.0   36.5    45.8     41.3     21.9     10.9      5.8   
p30-p50           0.9    5.9    15.3     29.6     3

# rt - da quantile: p0-p10 -33641 - -2712 would emcompass ~90% top 1% congestion loss days,  ~60% top 1%-5% loss days.

In [4]:
# new feature that rt-da smaller than -3000 as 1 and not -3000 as 0, we want to test split that would best predict 1 
for df in (train_df, holdout_df):
    df['bad_rtda'] = (df['rtm'] - df['dam'] < -3000).astype(int)
    
print('train_df :', train_df['bad_rtda'].value_counts().to_dict())
print('holdout_df:', holdout_df['bad_rtda'].value_counts().to_dict())

train_df : {0: 601707, 1: 2014}
holdout_df: {0: 230920, 1: 804}


# merge with physical variables and the dfax related variables 

In [5]:
# merge daily physical variables (wind / load / genoutage / ice price forecast) onto the splits
import sys
sys.path.append("/var/www/python/Prod/nighthawk/")
import pandas as pd, numpy as np
from nighthawk.data.pipeline.common_functions import wind, load, genoutage
from nighthawk.data.pipeline.var_handler import ice_elec_price_vh

OPEX = 'SPP'

# date range covering both splits
dts = pd.concat([train_df['dt'], holdout_df['dt']]).astype(str)
start_dt, end_dt = dts.min(), dts.max()
print('phys date range:', start_dt, '->', end_dt)

def _daily(df, col, name):
    """hourly forecast -> daily mean, dt as 'YYYY-MM-DD' string"""
    df = df.copy()
    df['dt'] = pd.to_datetime(df['dt']).dt.strftime('%Y-%m-%d')
    return df.groupby('dt', as_index=False)[col].mean().rename(columns={col: name})

# --- wind / load / genoutage daily forecast ---
wind_df  = wind.Wind(OPEX).get_total_wind(start_dt, end_dt, var_spec=['f'], impute=True)
wind_daily = _daily(wind_df, 'spp_wind_total_forecast_f', 'wind_forecast')

load_df  = load.Load(OPEX).get_total_load(start_dt, end_dt, var_spec=['f'], impute=True)
load_daily = _daily(load_df, 'spp_load_total_forecast_f', 'load_forecast')

go_df    = genoutage.GenOutage(OPEX).get_genoutage_by_level(start_dt, end_dt, var_spec=['f'], area_list=['SPP'])
go_col   = [c for c in go_df.columns if c.endswith('_forecast_f')][0]
genoutage_daily = _daily(go_df, go_col, 'genoutage_forecast')

# --- ice price forecast (INDIANAHUB proxy for SPP, node 636) ---
ice_df, _ = ice_elec_price_vh.get_data_and_mapping_for_ice_elec(
    [636], OPEX, ['INDIANAHUB'], start_dt, end_dt, var_spec=['f'], impute=True)
ice_daily = _daily(ice_df, 'INDIANAHUB_ice_elec_price_forecast_f', 'ice_price_forecast')

# --- combine all daily physical variables ---
phys_daily = (wind_daily
              .merge(load_daily,      on='dt', how='outer')
              .merge(genoutage_daily, on='dt', how='outer')
              .merge(ice_daily,       on='dt', how='outer'))
print('phys_daily:', phys_daily.shape)
display(phys_daily.head())

# --- merge onto train / holdout (dt-level) ---
train_df['dt']   = train_df['dt'].astype(str)
holdout_df['dt'] = holdout_df['dt'].astype(str)
train_df   = train_df.merge(phys_daily, on='dt', how='left')
holdout_df = holdout_df.merge(phys_daily, on='dt', how='left')
print('train_df  :', train_df.shape, '| holdout_df:', holdout_df.shape)
train_df.head()

phys date range: 2021-06-02 -> 2026-05-19
phys_daily: (1813, 5)


,dt,wind_forecast,load_forecast,genoutage_forecast,ice_price_forecast
0,2021-06-02,1916.375000,26712.708333,11762.216667,26.790000
1,2021-06-03,3768.707917,28603.416667,11286.708333,28.130000
2,2021-06-04,9523.051667,30291.458333,11359.608333,29.570000
3,2021-06-05,13756.800417,29792.250000,11398.025000,29.783333
4,2021-06-06,13693.260000,29196.708333,11431.691667,34.360000


train_df  : (603721, 87) | holdout_df: (231724, 85)


,constraint_family_num,dt,profit,profit_congestion,long_profit,short_profit,long_profit_sum_3d,short_profit_sum_3d,long_profit_sum_7d,short_profit_sum_7d,...,KV,KV_group,monitored_name,short_profit_q,rt_da_q,bad_rtda,wind_forecast,load_forecast,genoutage_forecast,ice_price_forecast
0,7,2021-07-01,14353.11877,5089.66448,0.0,0.0,0.00000,0.0,0.000000,0.000000,...,161.0,04_345,ln70_bluf-sub1214,NaN,NaN,0,3450.68,34594.75,8961.054167,35.99
1,11,2021-07-01,3893.43353,-828.00122,0.0,0.0,0.00000,0.0,0.000000,0.000000,...,161.0,04_345,lnadabell-vbi5,NaN,NaN,0,3450.68,34594.75,8961.054167,35.99
2,20,2021-07-01,2606.85200,82.42028,0.0,0.0,-91.79301,0.0,-1198.835983,224.251705,...,NaN,03_138,lnallen2-quaker,NaN,NaN,0,3450.68,34594.75,8961.054167,35.99
3,27,2021-07-01,2922.76579,160.79166,0.0,0.0,0.00000,0.0,0.000000,0.000000,...,NaN,03_138,lnanadarko-pocaset,NaN,NaN,0,3450.68,34594.75,8961.054167,35.99
4,28,2021-07-01,10621.29462,2009.66464,0.0,0.0,0.00000,0.0,0.000000,0.000000,...,138.0,03_138,lnanadarko-sequoyah,NaN,NaN,0,3450.68,34594.75,8961.054167,35.99


In [6]:
# load (drop the unnamed index col) and align the key name
dfax = (pd.read_csv(file_loc + 'dfax_variable.csv', index_col=0)
          .rename(columns={'constraintFamilyNum': 'constraint_family_num'}))
dfax['dt'] = dfax['dt'].astype(str)
dfax['constraint_family_num'] = dfax['constraint_family_num'].astype('int64')

# dfax should be unique per (dt, family) so the left join can't duplicate rows
assert not dfax.duplicated(['dt', 'constraint_family_num']).any(), 'dfax has dup (dt,family) keys'

# align the key dtypes on both frames, then left-join
for df in (train_df, holdout_df):
    df['dt'] = df['dt'].astype(str)
    df['constraint_family_num'] = df['constraint_family_num'].astype('int64')

train_df   = train_df.merge(dfax,   on=['dt', 'constraint_family_num'], how='left')
holdout_df = holdout_df.merge(dfax, on=['dt', 'constraint_family_num'], how='left')

print('train_df  :', train_df.shape,   '| dfax matched:', f"{train_df['dfax_max'].notna().mean()*100:.1f}%")
print('holdout_df:', holdout_df.shape, '| dfax matched:', f"{holdout_df['dfax_max'].notna().mean()*100:.1f}%")

train_df  : (603721, 98) | dfax matched: 100.0%
holdout_df: (231724, 96) | dfax matched: 100.0%


# Step 1: Shrink the dataset to have the constraints that has mvalue smaller than rt or da mvalue smaller than -500 



In [7]:
train_df['rt_da_smaller_500'] = ((train_df['rtm'] < -500) | (train_df['dam'] < -500)).astype(int)

In [8]:
# === 1. feature setup (family-day level) ===
KEYS    = ['dt', 'constraint_family_num']
TARGETS = ['rt_da_smaller_500']
# same-day realized / id columns -> EXCLUDE (leakage or non-features)
LEAK = ['dam', 'rtm', 'da_mvalue', 'rt_mvalue', 'bad_short', 'rt_da', 'rt_binds',
        'oops_constraint_num', 'monitored_clean', 'contingency_clean',
        'monitored_name', 'node_name',
       'short_profit_q', 'rt_da_q', 'bad_rtda','long_profit','short_profit','profit_congestion','profit']

num_cols = train_df.select_dtypes('number').columns.tolist()
features = [c for c in num_cols if c not in KEYS + TARGETS + LEAK]
CATS     = [c for c in ['KV_group'] if c in train_df.columns]   # categorical driver(s)

# tag features so you can separate "it was bad recently" from real conditions
persistence = [c for c in features if 'profit_sum' in c]
condition   = [c for c in features if c not in persistence]

print(f"{len(features)} numeric features (+{len(CATS)} categorical: {CATS})")
print(f"\nPERSISTENCE ({len(persistence)}): {persistence}")
print(f"\nCONDITION   ({len(condition)}): {condition}")

83 numeric features (+1 categorical: ['KV_group'])

PERSISTENCE (10): ['long_profit_sum_3d', 'short_profit_sum_3d', 'long_profit_sum_7d', 'short_profit_sum_7d', 'long_profit_sum_1m', 'short_profit_sum_1m', 'long_profit_sum_3m', 'short_profit_sum_3m', 'long_profit_sum_1y', 'short_profit_sum_1y']

CONDITION   (73): ['FlowRatio', 'ShadowPrice_min', 'ShadowPrice_sum', 'MinFlowLimit', 'MaxFlowLimit', 'dam_lag1', 'rtm_lag1', 'dam_sum_7d', 'rtm_sum_7d', 'dam_sum_1m', 'rtm_sum_1m', 'dam_sum_3m', 'rtm_sum_3m', 'dam_sum_1y', 'rtm_sum_1y', 'dam_sum_3y', 'rtm_sum_3y', 'rtda_1d', 'rtda_sum_7d', 'rtda_avg_7d', 'rtda_max_7d', 'rtda_min_7d', 'rtda_sum_1m', 'rtda_avg_1m', 'rtda_max_1m', 'rtda_min_1m', 'rtda_sum_3m', 'rtda_avg_3m', 'rtda_max_3m', 'rtda_min_3m', 'rtda_sum_1y', 'rtda_avg_1y', 'rtda_max_1y', 'rtda_min_1y', 'rtda_sum_3y', 'rtda_avg_3y', 'rtda_max_3y', 'rtda_min_3y', 'rtda_std_1m', 'rtda_std_3m', 'rtda_std_1y', 'rtda_std_3y', 'rt_max_7d', 'rt_max_1m', 'rt_max_3m', 'rt_max_1y', 'rt_max_3y', '

In [10]:
# === capture-the-1s: top-variable splits + tree rules for rt_da_smaller_500 ===
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.tree import DecisionTreeClassifier, _tree

TARGET = TARGETS[0]                              # 'rt_da_smaller_500'
FEATS  = globals().get('compact', features)      # compact set from step 1 if you ran it, else all features
y = train_df[TARGET].astype(int)
print(f'target={TARGET}  positives(y=1)={int(y.sum())} of {len(y)} ({y.mean()*100:.2f}%)')

# ---------- A. LightGBM classifier -> per-variable split threshold + side toward y=1 ----------
Xg = train_df[FEATS + CATS].copy()
for c in CATS: Xg[c] = Xg[c].astype('category')
clf = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                         subsample=0.8, colsample_bytree=0.8, class_weight='balanced',
                         random_state=0, n_jobs=-1, verbose=-1).fit(Xg, y, categorical_feature=CATS)
tdf = clf.booster_.trees_to_dataframe(); val = tdf.set_index('node_index')['value']
s = tdf[tdf['split_feature'].notna()].copy()
s['thr'] = pd.to_numeric(s['threshold'], errors='coerce')
s = s.dropna(subset=['thr']); s = s[s['thr'].abs() < 1e10]
s['side1'] = np.where(s['left_child'].map(val) > s['right_child'].map(val), '<=', '>')
def agg(d):
    gg, gl = d.loc[d.side1=='>','split_gain'].sum(), d.loc[d.side1=='<=','split_gain'].sum()
    return pd.Series({'n_splits': len(d), 'median_thr': round(d.thr.median(),3),
                      'y1_when': '>' if gg>=gl else '<=', 'dir_consistency': round(max(gg,gl)/d.split_gain.sum(),2),
                      'total_gain': round(d.split_gain.sum(),1)})
print('\n=== top variables: threshold + which side -> y=1 ===')
display(s.groupby('split_feature').apply(agg).sort_values('total_gain', ascending=False).head(12))

# ---------- B. shallow tree -> rules ranked by SHARE OF 1s CAPTURED (recall) ----------
Xt = train_df[FEATS + CATS].copy()
for c in CATS: Xt[c] = Xt[c].astype('category').cat.codes
feat = list(Xt.columns)
surr = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, class_weight='balanced',
                              random_state=0).fit(Xt, y)
dp = surr.decision_path(Xt)
n1 = np.asarray(dp[y.values==1].sum(0)).ravel(); n0 = np.asarray(dp[y.values==0].sum(0)).ravel()
t, TOT1 = surr.tree_, int(y.sum())
def rules(tree):
    tt, out = tree.tree_, {}
    def rec(n, c):
        if tt.feature[n]==_tree.TREE_UNDEFINED: out[n]=' AND '.join(c) or '(all)'
        else:
            f, th = feat[tt.feature[n]], tt.threshold[n]
            rec(tt.children_left[n], c+[f'{f}<={th:.3f}']); rec(tt.children_right[n], c+[f'{f}>{th:.3f}'])
    rec(0, []); return out
R = rules(surr); leaves=[i for i in range(t.node_count) if t.feature[i]==_tree.TREE_UNDEFINED]
tab = pd.DataFrame([{'pct_of_1s': round(n1[i]/TOT1*100,1), 'purity_%': round(n1[i]/(n1[i]+n0[i])*100,1),
                     'n_1': int(n1[i]), 'n_rows': int(n1[i]+n0[i]), 'rule': R[i]} for i in leaves])
print('\n=== leaf rules ranked by SHARE OF 1s CAPTURED (recall) ===')
display(tab.sort_values('pct_of_1s', ascending=False))

# ---------- greedy: take highest-purity leaves first -> cumulative capture vs rows touched ----------
cum = tab.sort_values('purity_%', ascending=False).copy()
cum['cum_pct_of_1s'] = (cum['n_1'].cumsum()/TOT1*100).round(1)
cum['cum_pct_rows']  = (cum['n_rows'].cumsum()/len(y)*100).round(1)
print('\n=== cut the highest-purity leaves first: cumulative 1s captured vs days touched ===')
display(cum[['rule','purity_%','n_1','cum_pct_of_1s','cum_pct_rows']])


target=rt_da_smaller_500  positives(y=1)=19157 of 603721 (3.17%)

=== top variables: threshold + which side -> y=1 ===


,n_splits,median_thr,y1_when,dir_consistency,total_gain
split_feature,,,,,
rtda_std_1m,225,36.710,>,1.00,3055578.9
rt_max_1m,41,-933.854,<=,1.00,389994.8
FlowRatio,712,0.719,>,0.98,372647.4
rtm_sum_7d,125,-1069.750,<=,0.99,269510.8
rtm_sum_1m,92,-3942.572,<=,0.99,168173.0
dam_sum_7d,84,-534.166,<=,0.99,153959.3
rt_max_7d,54,-140.423,<=,0.99,110640.9
wind_forecast,568,11957.353,>,0.91,93586.5
rtda_std_3m,171,92.239,>,0.93,60232.2



=== leaf rules ranked by SHARE OF 1s CAPTURED (recall) ===


,pct_of_1s,purity_%,n_1,n_rows,rule
9,35.0,49.2,6696,13616,rtda_std_1m>91.026 AND rt_max_7d<=-105.079 AND...
8,16.5,21.9,3152,14409,rtda_std_1m>91.026 AND rt_max_7d<=-105.079 AND...
11,12.0,18.3,2292,12505,rtda_std_1m>91.026 AND rt_max_7d<=-105.079 AND...
10,8.4,8.5,1617,19105,rtda_std_1m>91.026 AND rt_max_7d<=-105.079 AND...
14,8.0,14.1,1537,10891,rtda_std_1m>91.026 AND rt_max_7d>-105.079 AND ...
15,3.5,4.1,676,16644,rtda_std_1m>91.026 AND rt_max_7d>-105.079 AND ...
13,2.9,6.0,564,9440,rtda_std_1m>91.026 AND rt_max_7d>-105.079 AND ...
12,2.8,1.6,531,32246,rtda_std_1m>91.026 AND rt_max_7d>-105.079 AND ...
7,2.5,2.2,484,21548,rtda_std_1m<=91.026 AND rtda_std_1m>0.979 AND ...
3,1.6,0.2,304,145719,rtda_std_1m<=91.026 AND rtda_std_1m<=0.979 AND...



=== cut the highest-purity leaves first: cumulative 1s captured vs days touched ===


,rule,purity_%,n_1,cum_pct_of_1s,cum_pct_rows
9,rtda_std_1m>91.026 AND rt_max_7d<=-105.079 AND...,49.2,6696,35.0,2.3
8,rtda_std_1m>91.026 AND rt_max_7d<=-105.079 AND...,21.9,3152,51.4,4.6
11,rtda_std_1m>91.026 AND rt_max_7d<=-105.079 AND...,18.3,2292,63.4,6.7
14,rtda_std_1m>91.026 AND rt_max_7d>-105.079 AND ...,14.1,1537,71.4,8.5
6,rtda_std_1m<=91.026 AND rtda_std_1m>0.979 AND ...,12.2,174,72.3,8.8
10,rtda_std_1m>91.026 AND rt_max_7d<=-105.079 AND...,8.5,1617,80.7,11.9
2,rtda_std_1m<=91.026 AND rtda_std_1m<=0.979 AND...,6.7,92,81.2,12.1
13,rtda_std_1m>91.026 AND rt_max_7d>-105.079 AND ...,6.0,564,84.2,13.7
15,rtda_std_1m>91.026 AND rt_max_7d>-105.079 AND ...,4.1,676,87.7,16.5
4,rtda_std_1m<=91.026 AND rtda_std_1m>0.979 AND ...,2.8,240,88.9,17.9


Given the tree result, rtm_sum_1m<0 and many correlated ones give similar results that this would give the most coverage. 

In [11]:

total = train_df.query('short_profit<0').short_profit.sum()
now = train_df.query('rtm_sum_1m<-200 and short_profit<0 and rt_spike_3y').short_profit.sum()
display('profit coverage is about: ', now/total)
total_loss =  train_df.query(' short_profit<-400000').short_profit.count()
total_loss_100000 =  train_df.query(' short_profit<-100000').short_profit.count()

now_loss = train_df.query('rtm_sum_1m<-200 and short_profit<-400000').short_profit.count()
now_loss_100000 = train_df.query('rtm_sum_1m<-200 and short_profit<-100000').short_profit.count()

display(total_loss)
display(total_loss_100000)
display(now_loss)
display(now_loss_100000)

'profit coverage is about: '

np.float64(0.9085922128516756)

np.int64(227)

np.int64(1465)

np.int64(218)

np.int64(1328)

In [12]:
total = train_df.query('short_profit<0').short_profit.sum()
now = train_df.query('rtm_sum_1m<-200 and short_profit<0').short_profit.sum()
display('profit coverage is about: ', now/total)
total_loss =  train_df.query(' short_profit<-400000').short_profit.count()
total_loss_100000 =  train_df.query(' short_profit<-100000').short_profit.count()

now_loss = train_df.query('rtm_sum_1m<-200 and short_profit<-400000').short_profit.count()
now_loss_100000 = train_df.query('rtm_sum_1m<-200 and short_profit<-100000').short_profit.count()

display('profit loss under 100000', now_loss_100000/total_loss_100000)
display('profit loss under 400000', now_loss/total_loss)

display(total_loss)
display(total_loss_100000)
display(now_loss)
display(now_loss_100000)

display(len(train_df.query('rtm_sum_1m<-200')))

'profit coverage is about: '

np.float64(0.9085922128516756)

'profit loss under 100000'

np.float64(0.9064846416382253)

'profit loss under 400000'

...

np.int64(227)

np.int64(1465)

np.int64(218)

np.int64(1328)

133430

In [13]:
total = len(train_df[(train_df['rtm']-train_df['dam'])<-3000])
now = len(train_df.query('rtm_sum_1m<-200')[(train_df['rtm']-train_df['dam'])<-3000])
display(len(train_df.query('rtm_sum_1m<-200')[(train_df['rtm']-train_df['dam'])<-3000]))
display(len(train_df[(train_df['rtm']-train_df['dam'])<-3000]))
print(now/total)

/tmp/ipykernel_2778232/3155183448.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  now = len(train_df.query('rtm_sum_1m<-200')[(train_df['rtm']-train_df['dam'])<-3000])
/tmp/ipykernel_2778232/3155183448.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  display(len(train_df.query('rtm_sum_1m<-200')[(train_df['rtm']-train_df['dam'])<-3000]))


1788

2014

0.887785501489573


In [16]:
train_df.query('short_profit<-400000 and rtm_sum_1m>=-200').query('short_profit_q=="p0-p1" ').iloc[:,60:80]

,rtda_std_1y,rtda_std_3y,rt_max_7d,rt_max_1m,rt_max_3m,rt_max_1y,rt_max_3y,rt_nz_cnt_7d,rt_nz_cnt_1m,rt_nz_cnt_3m,rt_nz_cnt_1y,rt_nz_cnt_3y,rt_spike_7d,rt_spike_1m,rt_spike_3m,rt_spike_1y,rt_spike_3y,KV,KV_group,monitored_name
76338,57.200970,486.584172,-81.3018,-81.3018,-81.3018,-81.3018,-16044.8527,1.0,1.0,1.0,1.0,2.0,-69.687257,-78.591740,-80.398447,-81.079055,-8048.350168,138.0,03_138,lnquail-skyln
130025,385.078430,318.625451,0.0000,0.0000,-1479.3776,-6296.5954,-7069.1542,0.0,0.0,4.0,11.0,27.0,0.000000,0.000000,-791.239249,-1111.793772,-893.064378,161.0,04_345,lnmusktap-gore1
152655,948.272307,547.523268,0.0000,0.0000,-9863.2339,-9863.2339,-9863.2339,0.0,0.0,6.0,8.0,8.0,0.000000,0.000000,-2583.913889,-2489.055472,-2526.240241,138.0,03_138,lntulsa_no-cid_tap
182541,1689.194842,1077.198403,0.0000,0.0000,0.0000,-15845.0118,-15845.0118,0.0,0.0,0.0,26.0,75.0,0.000000,0.000000,0.000000,-4565.275949,-2300.839260,138.0,03_138,lntupelo2-sbrown
307877,1617.019256,934.456733,0.0000,0.0000,-11622.1695,-16765.2365,-16765.2365,0.0,0.0,2.0,17.0,17.0,0.000000,0.000000,-11264.862693,-5252.952639,-5424.025809,115.0,02_<=115,lnredbluff-rdrunner
341137,1252.608004,908.890147,-0.4881,-22.5867,-13813.8325,-13813.8325,-13813.8325,2.0,3.0,39.0,57.0,208.0,-0.174571,-6.922650,-889.462883,-1129.037970,-816.315582,NaN,03_138,multi-elementconstraint.spsnmties.spsnmties
350213,54.505792,277.178147,0.0000,0.0000,0.0000,0.0000,-9320.9179,0.0,0.0,0.0,0.0,9.0,0.000000,0.000000,0.000000,0.000000,-2436.432449,115.0,02_<=115,lnmidj-lec
366495,583.780073,388.196314,-44.8470,-47.1804,-311.4361,-7868.5127,-7868.5127,2.0,7.0,14.0,114.0,319.0,-18.670536,-16.527669,-30.693734,-259.629844,-200.483638,NaN,03_138,multi-elementconstraint.sppspsties.sppspsties
407856,1408.637478,932.187807,0.0000,0.0000,-4800.2700,-17772.5945,-17772.5945,0.0,0.0,17.0,107.0,142.0,0.000000,0.000000,-1586.320488,-1555.903054,-1822.672480,345.0,04_345,xfmrseminole-seminole


# Dayzer variable, KV and DFAX check 

In [17]:
def print_coverage(df, label):
    n = len(df); nfam = df['constraint_family_num'].nunique()
    for col, name in [('FlowRatio', 'Dayzer FlowRatio'), ('KV', 'KV-line')]:
        fam_cov = df.groupby('constraint_family_num')[col].apply(lambda s: s.notna().any()).sum()
        row_cov = int(df[col].notna().sum())
        print(f'[{label}] {name} coverage: {fam_cov}/{nfam} families '
              f'({fam_cov/nfam*100:.1f}%), {row_cov:,}/{n:,} rows ({row_cov/n*100:.1f}%)')

print_coverage(train_df, 'full')
print_coverage(train_df.query('rtm_sum_1m < -200'), 'rtm_sum_1m<-200')

def print_coverage(df, label):
    n=len(df); nfam=df['constraint_family_num'].nunique()
    print(f'[{label}] rows={n:,}  families={nfam}')
    for col,name in [('FlowRatio','Dayzer FlowRatio'),('KV','KV-line')]:
        fam=int(df.groupby('constraint_family_num')[col].apply(lambda s: s.notna().any()).sum())
        row=int(df[col].notna().sum())
        print(f'   {name:16s}: {fam}/{nfam} families ({fam/nfam*100:.1f}%), {row:,}/{n:,} rows ({row/n*100:.1f}%)')

print_coverage(train_df.query('rt_da < -3000'), 'rt_da < -3000')



[full] Dayzer FlowRatio coverage: 498/722 families (69.0%), 440,882/603,721 rows (73.0%)
[full] KV-line coverage: 492/722 families (68.1%), 448,455/603,721 rows (74.3%)
[rtm_sum_1m<-200] Dayzer FlowRatio coverage: 489/706 families (69.3%), 114,023/133,430 rows (85.5%)
[rtm_sum_1m<-200] KV-line coverage: 486/706 families (68.8%), 116,730/133,430 rows (87.5%)
[rt_da < -3000] rows=2,014  families=309
   Dayzer FlowRatio: 253/309 families (81.9%), 1,803/2,014 rows (89.5%)
   KV-line         : 257/309 families (83.2%), 1,854/2,014 rows (92.1%)


In [18]:
def print_coverage(df, label):
    n = len(df); nfam = df['constraint_family_num'].nunique()
    dz = df['FlowRatio'].notna()                                    # dayzer-covered rows
    print(f'[{label}] rows={n:,}  families={nfam}  dayzer rows={dz.mean()*100:.1f}%')
    for col, name in [('FlowRatio','Dayzer FlowRatio'), ('KV','KV-line')]:
        fam = int(df.groupby('constraint_family_num')[col].apply(lambda s: s.notna().any()).sum())
        row = int(df[col].notna().sum())
        print(f'   {name:16s}: {fam}/{nfam} fam ({fam/nfam*100:.1f}%), {row:,} rows ({row/n*100:.1f}%)')
    for col in ['rtm', 'dam']:                                      # value-weighted mvalue coverage
        val = df.loc[dz, col].sum() / df[col].sum() * 100
        print(f'   {col} value coverage by dayzer: {val:.1f}%')

print_coverage(train_df, 'full')
print_coverage(train_df.query('rt_da < -3000'), 'rt_da < -3000')


[full] rows=603,721  families=722  dayzer rows=73.0%
   Dayzer FlowRatio: 498/722 fam (69.0%), 440,882 rows (73.0%)
   KV-line         : 492/722 fam (68.1%), 448,455 rows (74.3%)
   rtm value coverage by dayzer: 91.0%
   dam value coverage by dayzer: 93.8%
[rt_da < -3000] rows=2,014  families=309  dayzer rows=89.5%
   Dayzer FlowRatio: 253/309 fam (81.9%), 1,803 rows (89.5%)
   KV-line         : 257/309 fam (83.2%), 1,854 rows (92.1%)
   rtm value coverage by dayzer: 90.8%
   dam value coverage by dayzer: 95.9%


In [20]:
# print('simulated_profit', train_df.short_profit.sum()+train_df.long_profit.sum()+holdout_df.short_profit.sum()+holdout_df.long_profit.sum())
# print('profit_congestion on bids table', bids.profit_congestion.sum())

# Constraint Analysis on the <-3000 constraints 

In [35]:
interested = train_df.query('rt_da < -3000 and short_profit<-400000 and profit_congestion < 0')
cols = ['monitored_name','dt','profit_congestion','long_profit','short_profit']
interested.monitored_name.value_counts()[:20]
# interested[interested['monitored_name']=='lnchar_ck-watford']

monitored_name
lnchar_ck-watford                              8
multi-elementconstraint.spsnmties.spsnmties    6
lnweav-tallgras                                6
lnrussett-sbrown                               4
lnsnakeck-aliance                              3
xfmrcimarron-cimarron                          3
lnosage_og-webbtap4                            3
xfmrseminole-seminole                          2
lngracmont-anadarko                            2
lncimarron-ckhal                               2
lnelliottw-enderlnw                            2
lnquail-skyln                                  2
lnstonewsw-tupelo2                             2
lnpenn1-sntfe                                  2
lnhobbs-cunnsub                                2
lndvisn-mstng1                                 2
xfmrturk_pp-turk_pp                            2
xfmredwv-edwv                                  2
lnsthrd-roman                                  1
xfmrkell-kell                                  1
Name:

In [ ]:
interested[interested['monitored_name']=='lnchar_ck-watford'].iloc[:,:20]

,constraint_family_num,dt,profit,profit_congestion,long_profit,short_profit,long_profit_sum_3d,short_profit_sum_3d,long_profit_sum_7d,short_profit_sum_7d,...,dfax_min,dfax_mean,dfax_std,dfax_absmean,dfax_pos_cnt,dfax_neg_cnt,dfax_strong,dfax_range,dfax_absmax,rt_da_smaller_500
234043,146,2023-08-15,936.29274,-2215.23562,533630.377701,-7.460579e+05,2.433463e+05,-4.665601e+05,28383.654501,-2.602902e+05,...,-0.0219,0.012522,0.125365,0.048743,7,85,0.076087,0.6938,0.6719,1
237005,146,2023-08-28,-264518.73827,-236924.75386,257042.516256,-4.173574e+06,-5.777034e+05,2.041292e+05,-495485.209537,1.647503e+05,...,-0.0219,0.005152,0.119245,0.042504,4,88,0.043478,0.6938,0.6719,1
241906,146,2023-09-08,-22588.14128,-17123.26865,220162.066380,-4.906260e+05,-2.836578e+05,-9.462928e+04,730905.021103,4.109795e+05,...,-0.0597,0.008048,0.111124,0.043032,15,95,0.127273,0.7316,0.6719,1
244406,146,2023-09-13,-14691.94440,-9947.34000,489532.354756,-6.041051e+05,1.603895e+05,6.162012e+05,-147851.233057,2.298330e+05,...,-0.0597,0.029560,0.146726,0.061846,16,56,0.166667,0.7316,0.6719,1
246839,146,2023-09-18,-375.34929,-6688.90157,690801.657852,-5.391679e+05,1.153274e+06,-1.992413e+06,883108.791550,-1.819607e+06,...,-0.0597,0.019798,0.149165,0.057678,7,51,0.103448,0.7316,0.6719,1
284197,146,2024-01-02,-72432.99157,-50239.53436,287844.867412,-7.863168e+05,-1.648572e+05,1.263785e+05,371217.999608,-2.819216e+05,...,-0.0219,0.001428,0.108772,0.038821,4,83,0.045977,0.6938,0.6719,1
291012,146,2024-01-24,-99335.39071,-112581.46230,661773.410999,-1.184170e+06,7.728766e+05,-2.185115e+05,815190.627367,1.992535e+05,...,-0.0219,0.007479,0.111952,0.042985,10,102,0.089286,0.6938,0.6719,1
306914,146,2024-02-25,-27478.35621,-36624.50334,599119.029747,-5.368825e+05,3.244092e+05,-1.905863e+05,241135.037074,8.135333e+05,...,-0.0597,-0.008125,0.093136,0.032557,2,108,0.036364,0.7316,0.6719,1


In [ ]:
feature	subset median	pctile in full	signal
rtm	−9,454	0.1	LOW — deep RT congestion
ShadowPrice_sum	−411	2.0	LOW — hard binding
rtm_sum_1m	−21,878	2.1	LOW
rt_max_1m	−7,224	2.3	LOW
ShadowPrice_min	−131	2.6	LOW
rtm_lag1	−187	3.2	LOW
rt_spike_3m	−966	11.7	LOW
rt_spike_1y	−1,293	18.9	LOW
dam	−1,735	0.7	LOW
wind_forecast	16,555	66.8	~mid (mildly high)
FlowRatio	1.00	69.1	~mid (but absolute ≈1 = binding)
rtda_std_1m	1,525	97.5	HIGH — volatile RT-DA basis

## Driver analysis — what moves long & short profit (family-day level)

Goal: rank the variables that drive **short_profit** (loss side) and **long_profit**, both as a
*level* (regression) and as *tail-loss events* (`short_profit < THRESH`), then translate the top
stable drivers into cut rules. Importance is judged on the **holdout** split (out-of-sample) and
cross-checked by SHAP **and** permutation importance — only drivers that agree are trusted.

**Leakage guard:** same-day realized `dam`/`rtm` are excluded (they are contemporaneous with the
loss). Only lagged/rolling history and pre-bid forecasts (FlowRatio, wind/load/genoutage/ice) are used.

In [40]:
train_df_filter = train_df.query('rtm_sum_1m<-100')

In [42]:
import lightgbm as lgb, numpy as np, pandas as pd
from sklearn.tree import DecisionTreeRegressor, export_text

d = train_df_filter.dropna(subset=['rt_da_q']).copy()      # rt_da<0 rows (band defined), within your rtm_sum_1m<-100 filter
y = d['rt_da_q'].cat.codes                            # 0=p0-p1 (worst) ... 9=p99-p100 (mildest)
FEATS = globals().get('compact', features)
X = d[FEATS + CATS].copy()
for c in CATS: X[c] = X[c].astype('category')
print('rows:', len(d), '| bands:', d['rt_da_q'].value_counts().sort_index().to_dict())

m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, min_child_samples=50,
                      subsample=0.8, colsample_bytree=0.8, random_state=0, n_jobs=-1, verbose=-1)
m.fit(X, y, categorical_feature=CATS)
imp = pd.Series(m.booster_.feature_importance('gain'), index=X.columns)
print((imp/imp.sum()).sort_values(ascending=False).head(15).round(4))

# best split point per feature
tdf = m.booster_.trees_to_dataframe(); s = tdf[tdf.split_feature.notna()].copy()
s['thr']=pd.to_numeric(s.threshold,errors='coerce'); s=s.dropna(subset=['thr']); s=s[s.thr.abs()<1e10]
print(s.groupby('split_feature').apply(lambda g: pd.Series(
    {'gain_wtd_thr':round(np.average(g.thr,weights=g.split_gain),3),'total_gain':round(g.split_gain.sum(),1)}))
    .sort_values('total_gain',ascending=False).head(12))

# readable tree
Xt = d[FEATS + CATS].copy()
for c in CATS: Xt[c]=Xt[c].astype('category').cat.codes
surr = DecisionTreeRegressor(max_depth=3, min_samples_leaf=200, random_state=0).fit(Xt, y)
print(export_text(surr, feature_names=list(Xt.columns), max_depth=3))


rows: 19554 | bands: {'p0-p1': 212, 'p1-p5': 823, 'p5-p10': 1022, 'p10-p30': 4055, 'p30-p50': 3988, 'p50-p70': 3907, 'p70-p90': 3748, 'p90-p95': 910, 'p95-p99': 713, 'p99-p100': 176}
rt_spike_1y           0.2114
rt_spike_3y           0.0713
rtda_std_1m           0.0686
wind_forecast         0.0361
rt_spike_3m           0.0288
rtm_lag1              0.0251
FlowRatio             0.0210
rt_spike_1m           0.0207
ice_price_forecast    0.0178
ShadowPrice_sum       0.0160
rt_max_1m             0.0149
ShadowPrice_min       0.0146
genoutage_forecast    0.0137
dfax_std              0.0130
rtm_sum_7d            0.0127
dtype: float64
                    gain_wtd_thr  total_gain
split_feature                               
rt_spike_1y             -642.288    103078.0
rt_spike_3y             -754.476     34761.9
rtda_std_1m             1178.313     33449.0
wind_forecast          16051.772     17581.0
rt_spike_3m             -581.634     14024.6
rtm_lag1               -1055.066     12251.7
FlowRat

In [43]:
import lightgbm as lgb, numpy as np, pandas as pd
from sklearn.tree import DecisionTreeRegressor, export_text

d = train_df_filter.dropna(subset=['bad_rtda']).copy()      # rt_da<0 rows (band defined), within your rtm_sum_1m<-100 filter
y = d['bad_rtda']                          # 0=p0-p1 (worst) ... 9=p99-p100 (mildest)
FEATS = globals().get('compact', features)
X = d[FEATS + CATS].copy()
for c in CATS: X[c] = X[c].astype('category')
print('rows:', len(d), '| bands:', d['bad_rtda'].value_counts().sort_index().to_dict())

m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, min_child_samples=50,
                      subsample=0.8, colsample_bytree=0.8, random_state=0, n_jobs=-1, verbose=-1)
m.fit(X, y, categorical_feature=CATS)
imp = pd.Series(m.booster_.feature_importance('gain'), index=X.columns)
print((imp/imp.sum()).sort_values(ascending=False).head(15).round(4))

# best split point per feature
tdf = m.booster_.trees_to_dataframe(); s = tdf[tdf.split_feature.notna()].copy()
s['thr']=pd.to_numeric(s.threshold,errors='coerce'); s=s.dropna(subset=['thr']); s=s[s.thr.abs()<1e10]
print(s.groupby('split_feature').apply(lambda g: pd.Series(
    {'gain_wtd_thr':round(np.average(g.thr,weights=g.split_gain),3),'total_gain':round(g.split_gain.sum(),1)}))
    .sort_values('total_gain',ascending=False).head(12))

# readable tree
Xt = d[FEATS + CATS].copy()
for c in CATS: Xt[c]=Xt[c].astype('category').cat.codes
surr = DecisionTreeRegressor(max_depth=3, min_samples_leaf=200, random_state=0).fit(Xt, y)
print(export_text(surr, feature_names=list(Xt.columns), max_depth=3))


rows: 142972 | bands: {0: 141158, 1: 1814}
rtm_lag1               0.0627
rtm_sum_7d             0.0488
wind_forecast          0.0435
FlowRatio              0.0306
ShadowPrice_min        0.0268
rtda_1d                0.0264
ice_price_forecast     0.0247
ShadowPrice_sum        0.0217
rtm_sum_1m             0.0203
short_profit_sum_3d    0.0183
rt_max_7d              0.0176
long_profit_sum_1m     0.0163
load_forecast          0.0160
dfax_std               0.0155
rt_spike_3y            0.0155
dtype: float64
                     gain_wtd_thr  total_gain
split_feature                                
rtm_lag1                -2087.965       584.8
rtm_sum_7d              -9185.039       455.5
wind_forecast           16734.833       406.0
FlowRatio                   0.880       285.1
ShadowPrice_min         -2670.097       249.9
rtda_1d                 -2019.953       246.5
ice_price_forecast         64.919       230.8
ShadowPrice_sum         -6435.059       202.5
rtm_sum_1m             -30261.45

In [45]:
from sklearn.tree import DecisionTreeRegressor, _tree
FEATS = [c for c in globals().get('compact', features) if c not in ('KV','KV_group')]

def kv_tree(g, grp):
    y = g['bad_rtda']; Xt = g[FEATS]
    surr = DecisionTreeRegressor(max_depth=3, min_samples_leaf=100, random_state=0).fit(Xt, y)
    imp = pd.Series(surr.feature_importances_, index=FEATS).sort_values(ascending=False)
    dp = surr.decision_path(Xt); yv = y.values
    n1 = np.asarray(dp[yv==1].sum(0)).ravel(); n0 = np.asarray(dp[yv==0].sum(0)).ravel()
    t, TOT1, ft = surr.tree_, int(yv.sum()), list(Xt.columns)
    print(f'\n== KV_group={grp}  rows={len(g)}  pos={TOT1}  base={yv.mean()*100:.2f}% ==')
    print('  top:', {k: round(v,2) for k,v in imp.head(5).items() if v>0})
    def show(n=0,d=0,b=''):
        tot=n1[n]+n0[n]; pur=n1[n]/tot*100 if tot else 0; rec=n1[n]/TOT1*100 if TOT1 else 0
        lab='LEAF' if t.feature[n]==_tree.TREE_UNDEFINED else f'[{ft[t.feature[n]]} <= {t.threshold[n]:.2f}]'
        print(f'  {"  "*d}{b}{lab}  pur={pur:4.1f}%  n1={int(n1[n])}  rec={rec:4.1f}%')
        if t.feature[n]!=_tree.TREE_UNDEFINED: show(t.children_left[n],d+1,'T>'); show(t.children_right[n],d+1,'F>')
    show()

for grp, g in train_df_filter.groupby('KV_group'):
    if len(g) >= 500 and g['bad_rtda'].sum() >= 30: kv_tree(g, grp)
    else: print(f'\n== KV_group={grp}: too few (rows={len(g)}, pos={int(g.bad_rtda.sum())}) ==')



== KV_group=02_<=115  rows=29176  pos=331  base=1.13% ==
  top: {'long_profit_sum_1m': 0.35, 'ShadowPrice_min': 0.21, 'dfax_std': 0.18, 'rtm_lag1': 0.12, 'rt_nz_cnt_3y': 0.08}
  [long_profit_sum_1m <= 911361.09]  pur= 1.1%  n1=331  rec=100.0%
    T>[ShadowPrice_min <= -1175.86]  pur= 0.9%  n1=271  rec=81.9%
      T>[rtda_min_1y <= -3721.58]  pur= 7.1%  n1=51  rec=15.4%
        T>LEAF  pur= 9.5%  n1=45  rec=13.6%
        F>LEAF  pur= 2.5%  n1=6  rec= 1.8%
      F>[rtm_lag1 <= -655.85]  pur= 0.8%  n1=220  rec=66.5%
        T>LEAF  pur= 3.7%  n1=65  rec=19.6%
        F>LEAF  pur= 0.6%  n1=155  rec=46.8%
    F>[dfax_std <= 0.18]  pur= 9.8%  n1=60  rec=18.1%
      T>[rt_nz_cnt_3y <= 102.50]  pur= 6.3%  n1=28  rec= 8.5%
        T>LEAF  pur= 0.5%  n1=1  rec= 0.3%
        F>LEAF  pur=10.4%  n1=27  rec= 8.2%
      F>LEAF  pur=18.9%  n1=32  rec= 9.7%

== KV_group=03_138  rows=45283  pos=714  base=1.58% ==
  top: {'rtm_lag1': 0.43, 'ShadowPrice_min': 0.21, 'wind_forecast': 0.09, 'rt_max_1m': 0.0